# 📗 검색 증강 생성(RAG) — 아키텍처와 벡터 검색의 필요성

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 시간엔 **LLM**(대규모 언어 모델)에게 프롬프트를 주고 답을 받아 봤습니다. 그런데 LLM 은 **배우지 않은 최신 정보**나 **우리 회사 내부 문서**는 알지 못하고, 모를 때조차 그럴듯하게 지어내는 **환각(hallucination)** 문제가 있습니다. 이번 시간엔 그 약점을 메우는 설계인 **RAG**(Retrieval-Augmented Generation, 검색 증강 생성)의 **구조**와, 그 심장인 **벡터 검색**이 왜 필요한지를 배웁니다. 실제 벡터 데이터베이스 구축은 **교안_02** 에서 이어서 합니다.

> 오늘은 **질문 → 검색 → 생성**까지 한 바퀴를 다 돕니다. 이 노트북과 교안_02 에서 **검색기**를 만들고, 교안_03 에서 찾아온 근거를 **LLM 에게 건네 답을 쓰게** 해 RAG 를 완성합니다.

## ⏪ 복습 — 지난 시간까지 쌓은 재료

RAG 는 새로운 마법이 아니라, 여러분이 이미 배운 조각들을 **이어 붙인 설계**입니다.

- **임베딩(지난 단원)**: `SentenceTransformer('jhgan/ko-sroberta-multitask')` 로 문장을 **768개의 숫자(벡터)** 로 바꿨습니다. 뜻이 비슷한 문장은 벡터도 가까이 놓입니다.
- **코사인 유사도(지난 단원)**: 두 벡터가 **얼마나 같은 방향**을 가리키는지를 잰 값(1에 가까울수록 비슷). `cosine_similarity(a, b)` 로 구했습니다.
- **LLM 프롬프트(지난 단원)**: OpenAI 클라이언트에 메시지를 주고 답을 받았습니다. LLM 은 아는 것은 잘 답하지만, **모르는 것**(최신·비공개 지식)은 못 답하거나 지어냅니다.

RAG 는 이 셋을 **검색 → 근거 → 생성** 순서로 엮습니다. 오늘 이 셋을 차례로 다 만들어 붙입니다.

**오늘의 목표**

- [ ] LLM 만으로는 왜 부족한지(**환각**·**최신성·비공개 지식**) 설명한다.
- [ ] RAG 의 큰 흐름 **질문(Query) → 검색기(Retriever) → 생성기(Generator)** 를 그림으로 그린다.
- [ ] RAG 를 이루는 **구성요소**(문서→임베딩→벡터 저장소→유사도 검색→컨텍스트→LLM)를 나열한다.
- [ ] 지난 단원의 코사인 유사도로 **직접 Top-K 검색기**를 만들어 본다.
- [ ] 문서가 많아지면 **선형 스캔**이 왜 느린지, 그래서 **벡터 데이터베이스**가 왜 필요한지 이해한다.
- [ ] 대표 솔루션 **Pinecone·ChromaDB·Qdrant** 의 성격 차이를 비교한다.

아래 셀을 먼저 실행해 라이브러리와 한국어 임베딩 모델을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 임베딩 모델을 준비합니다.
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

# 지난 단원에서 배운 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 바꿉니다.
# (처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.)
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.get_embedding_dimension())

## 데이터 살펴보기 — 여행지 소개 코퍼스

이번 시간엔 **여행지 소개** 문서를 씁니다. 관광지마다 이름(`name`)·지역(`region`)·유형(`type`: 해변/역사/자연/도시/미식)·입장료(`entrance_fee`, 원)와 한두 문장의 소개(`description`)가 붙어 있습니다. 전국 관광지 **100곳**이 유형별로 20곳씩 고르게 들어 있고, 이 소개 문장이 우리가 **검색할 문서**입니다. 새 데이터를 만나면 먼저 생김새부터 봅니다.

In [ ]:
# [제공 코드] 여행지 소개 데이터를 불러와 살펴봅니다.
spots = pd.read_csv('data/travel_spots.csv')

print("행·열 크기:", spots.shape)

# 유형이 고르게 섞였는지 확인
print("\n[유형별 개수] value_counts()"); print(spots["type"].value_counts())

# 지역은 나중에 '제주만 검색' 같은 필터에 쓸 메타데이터다
print("\n[지역별 개수]"); print(spots["region"].value_counts())

print("\n[앞 5행] head()"); display(spots.head())

## 1. 왜 RAG 인가 — LLM 의 두 가지 빈틈

LLM 은 방대한 글로 **미리 학습**된 모델입니다. 그래서 두 가지 약점이 있습니다.

1. **모르는 것을 지어낸다(환각)** — 학습에 없던 사실을 물으면, '모른다'가 아니라 그럴듯한 **거짓 답**을 만들어 내곤 합니다. 예: 우리 회사 환불 규정을 물으면 실제와 다른 규정을 자신 있게 답할 수 있습니다.
2. **최신·비공개 지식이 없다** — 모델이 학습을 마친 시점 **이후의 사건**이나, 사내 문서·개인 메모처럼 **학습 데이터에 없는 자료**는 알 수 없습니다.

**해결 아이디어**: 답하기 **전에**, 믿을 수 있는 문서 더미에서 질문과 관련된 대목을 **찾아와**(검색) LLM 에게 '이 근거를 바탕으로 답하라'고 건네면 됩니다. 이렇게 **검색으로 생성을 보강**하는 것이 바로 **RAG**(검색 증강 생성)입니다. 근거가 함께 있으니 환각이 줄고, 최신·사내 문서를 넣으면 그 지식까지 답에 반영됩니다.

> 오늘 배우는 **여행지 검색**은 RAG 의 '**찾아오기**' 부분입니다. 찾아온 문서를 LLM 에게 건네 **답을 쓰게** 하는 마지막 단계는 **교안_03** 에서 붙입니다.

### ✅ 바로 확인 퀴즈

1. LLM 이 학습하지 않은 사실을 물었을 때 그럴듯한 거짓을 지어내는 현상을 무엇이라 부르나요?
2. RAG 가 LLM 의 약점을 메우기 위해 **답을 생성하기 전에** 하는 일은 무엇인가요?

<details><summary>정답 보기</summary>

1. 환각(hallucination). 2. 질문과 관련된 문서(근거)를 **검색**해서 LLM 에게 함께 건넨다.

</details>

## 2. RAG 의 큰 그림 — 질문 → 검색기 → 생성기

RAG 파이프라인은 크게 세 부분입니다.

- **질문(Query)**: 사용자가 자연어로 묻습니다. 예: "바다에서 물놀이하기 좋은 곳 알려 줘"
- **검색기(Retriever)**: 질문을 임베딩해 **벡터 저장소**에서 의미가 가까운 문서 Top-K 를 찾아옵니다. 오늘의 주인공입니다.
- **생성기(Generator)**: 찾아온 문서를 **근거(컨텍스트)** 로 LLM 에게 건네 최종 답을 쓰게 합니다. (교안_03)

그림으로 보면 이렇습니다.

<img src="images/RAG_큰그림.png" width="960">

한 가지 더 — RAG 는 일이 벌어지는 **시점**이 둘로 나뉩니다.

- **인덱싱 타임(색인, 미리 준비)**: 가진 문서를 몽땅 임베딩해 벡터 저장소에 **넣어 둡니다**. 질문이 오기 전에 한 번 해 둡니다.
- **쿼리 타임(질문 시점)**: 질문이 들어오면 질문만 임베딩해 저장소에서 **가까운 문서를 꺼내** LLM 에 건넵니다.

<img src="images/인덱싱타임_vs_쿼리타임.png" width="960">

> **왼쪽(파랑) = 인덱싱 타임** — 질문이 오기 전에 문서를 몽땅 벡터로 바꿔 저장소에 넣어 둡니다. 시계 아이콘처럼 **미리 한 번** 해 두는 일입니다.
> **오른쪽(주황) = 쿼리 타임** — 질문이 들어올 때마다 그 질문만 벡터로 바꿔 저장소에서 가까운 것을 꺼내옵니다. 번개 아이콘처럼 **매번·빠르게** 일어나야 합니다.

> 이 두 시점은 **검색기**의 이야기입니다. 그렇게 찾아온 문서를 LLM 이 읽고 답을 쓰는 **생성**까지 가야 RAG 한 바퀴가 닫히고, 그건 교안_03 에서 합니다.

### ✅ 바로 확인 퀴즈

1. RAG 의 세 부분 중 **질문과 가까운 문서를 벡터 저장소에서 찾아오는** 부분의 이름은?
2. 문서를 미리 임베딩해 저장소에 넣어 두는 시점을 '인덱싱 타임'이라 합니다. 그러면 질문이 들어와 검색·답변하는 시점은 무엇이라 부를까요?

<details><summary>정답 보기</summary>

1. 검색기(Retriever). 2. 쿼리 타임(query time).

</details>

## 3. RAG 를 이루는 구성요소

검색기를 좀 더 잘게 쪼개면, 다음 부품들이 순서대로 이어집니다.

- **문서(Documents)**: 검색 대상 원문. 오늘은 여행지 소개 문장들입니다.
- **임베딩 모델(Embedding model)**: 문장을 768차원 벡터로 바꾸는 도구(지난 단원의 ko-sroberta).
- **벡터 저장소(Vector store)**: 문서 벡터를 담아 두고 **빠르게 최근접을 찾아 주는** 데이터베이스. 교안_02 에서 배울 ChromaDB 가 여기에 해당합니다.
- **유사도 검색(Similarity search)**: 질문 벡터와 가장 가까운 문서 벡터 Top-K 를 고르는 연산.
- **컨텍스트(Context)**: 찾아온 문서들을 모아 LLM 에게 건넬 근거 묶음.
- **LLM(생성기)**: 컨텍스트를 읽고 답을 쓰는 모델(교안_03 에서 OpenAI API 로 직접 붙입니다).

<img src="images/RAG_구성요소.png" width="960">

오늘 우리는 이 중 **문서 → 임베딩 → (직접 만든) 유사도 검색** 까지를 손으로 만들어 보고, **벡터 저장소**가 왜 따로 필요한지 체감합니다.

### ✅ 바로 확인 퀴즈

1. 문장을 768차원 숫자 벡터로 바꾸는 구성요소의 이름은?
2. 문서 벡터들을 담아 두고 질문과 가까운 것을 빠르게 찾아 주는 데이터베이스를 무엇이라 부르나요?

<details><summary>정답 보기</summary>

1. 임베딩 모델. 2. 벡터 저장소(벡터 데이터베이스, Vector store/DB).

</details>

## 4. 직접 만드는 검색기 — 코사인 유사도로 Top-K

벡터 저장소가 없어도, 지난 단원에서 배운 **코사인 유사도**만으로 검색기를 만들 수 있습니다. 방법은 단순합니다.

1. 모든 문서를 미리 임베딩해 둔다(인덱싱 타임).
2. 질문이 오면 질문을 임베딩한다.
3. 질문 벡터와 **모든 문서 벡터**의 코사인 유사도를 구한다.
4. 유사도가 높은 순으로 정렬해 **Top-K** 를 고른다.

아래 시연 셀에서 여행지 소개를 통째로 임베딩하고, 키워드가 겹치지 않는 질문으로 검색해 봅니다. 질문 "**바다에서 시원하게 물놀이하기 좋은 곳**" 에는 '해변'이라는 단어가 없지만, **의미가 가까운** 해변 문서가 위로 올라오는지 보세요.

**처음 보는 인자 하나 — `normalize_embeddings=True`**

임베딩은 768개의 숫자, 즉 원점에서 뻗어 나간 **화살표(벡터)** 입니다. 이 화살표는 **방향**(문장의 뜻)과 **길이**(크기)를 함께 갖는데, 검색에서 우리가 쓰는 건 **방향뿐**입니다. `normalize_embeddings=True` 는 모든 화살표의 **길이를 1로 맞춰 주는**(단위벡터로 만드는) 옵션입니다. 방향은 그대로 두고 길이만 통일하는 것이라 뜻은 조금도 변하지 않습니다.

굳이 왜 맞출까요. 코사인 유사도는 원래 `두 벡터의 내적 ÷ (길이 × 길이)` 인데, **길이가 모두 1이면 나눗셈이 사라져** 내적만 남습니다. 계산이 가벼워지고, 벡터 DB 가 쓰는 '코사인 거리' 와도 그대로 맞아떨어집니다. 그래서 이 단원에서는 **문서든 질문이든 임베딩할 때 항상 이 옵션을 켭니다.**

In [ ]:
# 1) 인덱싱 타임 — 모든 여행지 소개를 임베딩(문서 개수 × 768차원)
doc_texts = spots['description'].tolist()
doc_emb = emb_model.encode(doc_texts, normalize_embeddings=True)

print("문서 임베딩 행렬:", doc_emb.shape)

# 1.0 에 아주 가까우면 길이가 1로 잘 맞춰진 것이다
print("첫 문서 벡터의 길이:", round(float(np.linalg.norm(doc_emb[0])), 4))

문서 벡터가 준비됐습니다. 이제 **쿼리 타임** — 질문을 **같은 방식으로** 임베딩해 모든 문서와 견주고 Top-3 를 고릅니다.

In [ ]:
# 2~4) 쿼리 타임 — 질문도 문서와 같은 옵션으로 임베딩해야 같은 잣대가 된다
query = "바다에서 시원하게 물놀이하기 좋은 곳"

# encode 는 여러 문장을 받도록 만들어져, 한 문장이어도 리스트로 감싼다
query_emb = emb_model.encode([query], normalize_embeddings=True)

# 결과가 (1, 문서 수) 표라 [0] 으로 그 한 줄만 꺼낸다
sims = cosine_similarity(query_emb, doc_emb)[0]

# argsort 는 오름차순 인덱스라, 부호를 뒤집어 내림차순으로 만든다
top3 = np.argsort(-sims)[:3]

print("\n질문:", query)

for rank, i in enumerate(top3, 1):
    print(f"  {rank}위  유사도 {sims[i]:.3f}  |  {spots.loc[i, 'name']} ({spots.loc[i, 'type']})")

키워드('해변')가 없어도 **바다·물놀이와 의미가 가까운 해변 관광지**가 위로 올라옵니다. 이것이 **의미 기반 검색**의 힘입니다(다음 시간에 자세히).

### 🖐️ 함께 따라하기

위 시연을 참고해, 이번에는 질문을 "**옛 궁궐과 전통 건축을 보고 싶다**" 로 바꿔 **Top-3** 를 뽑아 보세요. `doc_emb` 는 위에서 만든 것을 그대로 씁니다.

1. 질문 문자열을 `my_query` 에 담고 `emb_model.encode([my_query], normalize_embeddings=True)` 로 질문 벡터를 만든다.
2. `cosine_similarity(질문벡터, doc_emb)[0]` 으로 유사도 배열을 구한다.
3. `np.argsort(-유사도)[:3]` 으로 상위 3개 인덱스를 골라, 각 관광지의 `name` 과 `type` 을 출력한다.

In [ ]:
# 여기에 위 1~3 단계를 직접 작성해 보세요.
# 힌트: 시연 셀의 query→query_emb→sims→top3 흐름을 그대로 따르되 질문만 바꿉니다.

### 그런데 — 문서가 많아지면?

위 방법은 질문마다 **모든 문서와 하나하나 유사도를 계산**합니다. 이것을 **선형 스캔(brute-force)** 이라 합니다. 문서가 100개면 순식간이지만, 실제 서비스는 문서가 **수십만~수억 개**입니다.

- 문서가 N개면 질문 한 번에 **N번**의 유사도 계산이 필요합니다(N에 비례해 느려짐).
- 벡터를 전부 메모리에 올려 두어야 하고, 새 문서 추가·삭제도 직접 관리해야 합니다.

말로만 하지 말고 **직접 재 봅시다.** 여행지 100건은 너무 적으니, 가진 벡터를 복사해 문서 수를 5,000건·50,000건으로 불린 뒤 같은 검색을 반복해 시간을 잽니다.

In [ ]:
import time

query_emb = emb_model.encode(['바다에서 시원하게 물놀이하기 좋은 곳'], normalize_embeddings=True)

# 문서 수를 바꿔 가며 같은 검색을 돌려 본다
for n_docs in [100, 5_000, 50_000]:
    # 내용이 아니라 '규모'의 효과를 보려고 같은 벡터를 복사해 늘린다
    big = np.repeat(doc_emb, n_docs // len(doc_emb) + 1, axis=0)[:n_docs]

    t0 = time.perf_counter()
    for _ in range(5):                       # 5번 재서 평균
        sims_big = cosine_similarity(query_emb, big)[0]
        _ = np.argsort(-sims_big)[:3]
    per_query = (time.perf_counter() - t0) / 5

    memory_mb = big.nbytes / 1024 / 1024
    print(f'문서 {n_docs:>7,}건 | 질문 1회 {per_query * 1000:7.2f}ms | 벡터가 차지한 메모리 {memory_mb:6.1f}MB')

문서가 **10배** 늘면 검색 시간도 대략 **10배**, 메모리도 **10배** 늘어납니다(100건처럼 아주 작을 때는 고정 비용이 커서 비율이 딱 맞지 않지만, 5,000건 → 50,000건 구간을 보면 거의 정확히 10배입니다). 수백만 건이면 질문 하나에 몇 초씩 걸리고, 벡터만으로 기가바이트를 차지합니다. **이것이 벡터 데이터베이스가 필요한 이유입니다.**

<img src="images/벡터DB가_하는_일.png" width="960">

그래서 실무에서는 **벡터 데이터베이스**를 씁니다. 벡터 DB 는 (1) 문서 벡터를 **저장·관리**하고, (2) 전부 비교하지 않고도 가까운 것을 빠르게 찾는 **근사 최근접 탐색(ANN)** 으로 검색 속도를 끌어올리며, (3) 유형·지역 같은 **메타데이터 필터**까지 함께 걸어 줍니다. 이 도구들은 **다음 시간**에 직접 만져 봅니다.

### ✅ 바로 확인 퀴즈

1. 질문마다 모든 문서와 일일이 유사도를 계산하는 방식을 무엇이라 부르나요?
2. 문서가 수백만 개로 늘어날 때 선형 스캔의 문제는 무엇인가요? (한 가지)

<details><summary>정답 보기</summary>

1. 선형 스캔(brute-force). 2. 문서 수 N에 비례해 계산량이 늘어 **검색이 느려진다**(그리고 모든 벡터를 직접 메모리에 올려 관리해야 한다).

</details>

## 5. 벡터 데이터베이스 솔루션 비교 — Pinecone · ChromaDB · Qdrant

벡터 DB 는 종류가 많습니다. 대표적인 셋의 성격을 비교하면 이렇습니다.

**ChromaDB**
- 성격: 파이썬에 바로 붙는 **가볍고 로컬 친화적인** 오픈소스 벡터 DB.
- 장점: 설치가 간단하고 메모리·로컬 저장 모드가 있어 **학습·프로토타입**에 좋다.
- 이 과정: 다음 시간부터 **ChromaDB 를 기본**으로 벡터 검색을 실습한다.

**Qdrant**
- 성격: Rust 로 만든 **오픈소스** 벡터 DB. 로컬(메모리)부터 서버·클라우드까지 확장된다.
- 장점: **프로덕션 급 성능**과 강력한 메타데이터 필터. 자체 호스팅이 가능하다.
- 이 과정: 다음 시간에 `:memory:` 모드로 **같은 코퍼스를 재현**해 ChromaDB 와 비교한다.

**Pinecone**
- 성격: 설치·운영이 필요 없는 **완전 관리형 클라우드** 벡터 DB(SaaS).
- 장점: 인프라를 신경 쓰지 않고 **대규모**를 맡길 수 있다. 다만 외부 서비스라 **API 키·비용**이 든다.
- 이 과정: 클라우드형의 **개념만** 소개한다(이 수업에서 코드로 다루지는 않는다).

<img src="images/솔루션_비교.png" width="960">

핵심은 **셋 다 하는 일은 같다** — 벡터를 저장하고, 질문 벡터와 가까운 문서를 빠르게 찾아 준다. '로컬이냐 클라우드냐', '가벼움이냐 프로덕션 성능이냐'의 **운영 성격**이 다를 뿐입니다.

### ✅ 바로 확인 퀴즈

1. 설치·서버 운영 없이 클라우드에 맡기는 **완전 관리형** 벡터 DB 는 셋 중 무엇인가요?
2. 이 과정에서 **다음 시간부터 기본으로** 실습할, 파이썬 친화적이고 가벼운 로컬 벡터 DB 는 무엇인가요?

<details><summary>정답 보기</summary>

1. Pinecone. 2. ChromaDB.

</details>

## 이번 강의 정리

- **RAG**(검색 증강 생성)는 LLM 의 **환각**과 **최신·비공개 지식 부재**를 메우려고, 답하기 전에 관련 문서를 **검색해 근거로 함께 건네는** 설계다.
- 큰 흐름은 **질문(Query) → 검색기(Retriever) → 생성기(Generator)**. 문서를 미리 임베딩해 넣는 **인덱싱 타임**과, 질문을 임베딩해 찾는 **쿼리 타임**으로 나뉜다.
- 코사인 유사도만으로 **직접 Top-K 검색기**를 만들 수 있지만, 문서가 많아지면 **선형 스캔**이 느려 **벡터 데이터베이스**(저장·ANN 검색·메타 필터)가 필요하다.
- 대표 솔루션은 **ChromaDB**(가벼운 로컬)·**Qdrant**(오픈소스 프로덕션)·**Pinecone**(관리형 클라우드).

## ⏭️ 예고 — 다음 시간: 벡터 검색과 벡터 DB 구축

다음 시간엔 **ChromaDB** 로 문서 벡터를 실제로 저장하고, 의미 기반 검색·**메타데이터 필터**·**ANN/HNSW** 개념을 익힙니다. **Qdrant** 로 같은 검색을 재현해 두 도구를 비교합니다. 여기서 완성한 검색기에 **LLM 생성**을 붙여 RAG 를 완성하는 것이 교안_03 입니다.